# 02 - EDA exploratorio

Este notebook inicia el analisis exploratorio del dataset de reservas hoteleras. El objetivo es entender la calidad de los datos, las distribuciones principales y la relacion inicial entre las variables y el target `booking_status`.

Este notebook aun no entrena modelos. Su funcion es ayudarnos a decidir como preparar los datos antes del baseline.

## 1. Preguntas guia del EDA

- Hay nulos o duplicados?
- El target esta balanceado?
- Que variables numericas tienen distribuciones raras u outliers?
- Que variables categoricas tienen muchas categorias?
- Que variables parecen relacionarse con la cancelacion?
- Hay columnas que debamos excluir o transformar antes del modelo?

## 2. Importacion de librerias

Usaremos `pandas` para manipular datos, `matplotlib` y `seaborn` para visualizar. Estas visualizaciones seran simples y orientadas a entender el dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

## 3. Carga del dataset

Leemos el archivo original desde `data/raw/`. En EDA podemos explorar, pero evitamos modificar el archivo crudo.

In [ ]:
DATA_PATH = Path("../data/raw/hotel-reservations-classification-dataset/Hotel Reservations.csv")
TARGET = "booking_status"

df = pd.read_csv(DATA_PATH)
df.head()

## 4. Revision general

Primero confirmamos dimensiones, columnas y tipos. Esta revision sirve para detectar si el archivo cargado coincide con lo esperado en la SPEC.

In [ ]:
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.info()

Interpretacion esperada:

- Esperamos 36.275 filas y 19 columnas.
- Si aparecen menos columnas o nombres distintos, debemos revisar la descarga del dataset.
- Los tipos detectados por pandas nos ayudaran a separar variables numericas y categoricas.

## 5. Nulos y duplicados

Los nulos pueden requerir imputacion. Los duplicados pueden inflar artificialmente el rendimiento del modelo si no se revisan.

In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary

In [ ]:
duplicated_rows = df.duplicated().sum()
print(f"Filas duplicadas exactas: {duplicated_rows}")

Interpretacion a completar tras ejecutar:

- Si no hay nulos, el preprocesamiento inicial sera mas simple.
- Si hay nulos, decidiremos si imputar, eliminar o crear una categoria especial.
- Si hay duplicados exactos, revisaremos si deben eliminarse antes del split.

## 6. Distribucion del target

Esta grafica muestra cuantas reservas fueron canceladas y cuantas no. Es importante porque una clase mayoritaria puede hacer que `accuracy` sea enganosa.

In [ ]:
target_counts = df[TARGET].value_counts()
target_percentages = df[TARGET].value_counts(normalize=True).mul(100).round(2)

display(pd.DataFrame({"count": target_counts, "percentage": target_percentages}))

ax = sns.countplot(data=df, x=TARGET, order=target_counts.index)
ax.set_title("Distribucion del target")
ax.set_xlabel("Estado de la reserva")
ax.set_ylabel("Numero de reservas")
plt.show()

Interpretacion confirmada desde la inspeccion inicial:

- `Not_Canceled` representa aproximadamente 67.2%.
- `Canceled` representa aproximadamente 32.8%.
- Hay desbalance moderado, pero no extremo.
- Para evaluar modelos necesitaremos mirar `precision`, `recall` y `F1-score`, no solo `accuracy`.

## 7. Separacion inicial de columnas

Separamos columnas numericas, categoricas, binarias y excluidas. Esta clasificacion inicial puede cambiar despues del EDA.

In [ ]:
excluded_columns = ["Booking_ID"]

categorical_columns = [
    "type_of_meal_plan",
    "room_type_reserved",
    "market_segment_type",
]

binary_columns = [
    "required_car_parking_space",
    "repeated_guest",
]

numeric_columns = [
    "no_of_adults",
    "no_of_children",
    "no_of_weekend_nights",
    "no_of_week_nights",
    "lead_time",
    "arrival_year",
    "arrival_month",
    "arrival_date",
    "no_of_previous_cancellations",
    "no_of_previous_bookings_not_canceled",
    "avg_price_per_room",
    "no_of_special_requests",
]

print(f"Numericas: {len(numeric_columns)}")
print(f"Categoricas: {len(categorical_columns)}")
print(f"Binarias: {len(binary_columns)}")
print(f"Excluidas: {excluded_columns}")

Interpretacion inicial:

- Las variables categoricas necesitaran encoding antes de entrenar modelos.
- Las variables numericas pueden requerir escalado segun el algoritmo usado.
- `Booking_ID` queda fuera para evitar memorizar identificadores.

## 8. Estadisticos de variables numericas

Con `describe()` revisamos media, desviacion, minimos, maximos y percentiles. Esto ayuda a detectar rangos raros u outliers.

In [ ]:
df[numeric_columns].describe().T

Interpretacion a completar tras ejecutar:

- Revisar si hay valores imposibles, por ejemplo precios negativos o cantidades de noches incoherentes.
- Prestar atencion a `lead_time` y `avg_price_per_room`, porque suelen tener outliers.
- Revisar si algunas variables tienen muchos ceros, como cancelaciones previas.

## 9. Distribuciones numericas principales

Visualizamos algunas variables numericas para entender su forma. No buscamos aun conclusiones finales, solo patrones iniciales.

In [ ]:
selected_numeric = [
    "lead_time",
    "avg_price_per_room",
    "no_of_week_nights",
    "no_of_weekend_nights",
    "no_of_special_requests",
]

df[selected_numeric].hist(figsize=(14, 8), bins=30)
plt.suptitle("Distribuciones numericas principales")
plt.tight_layout()
plt.show()

Interpretacion a completar tras ejecutar:

- Variables muy asimetricas pueden necesitar transformaciones o modelos robustos.
- Los outliers no se eliminan automaticamente; primero se interpretan.
- Si una variable concentra muchos valores en cero, puede seguir siendo util para clasificar.

## 10. Variables categoricas

Revisamos cuantas categorias tiene cada variable y si alguna categoria domina demasiado.

In [ ]:
for column in categorical_columns:
    print(f"\n{column}")
    display(df[column].value_counts(dropna=False))

Interpretacion a completar tras ejecutar:

- Si una variable tiene pocas categorias, One-Hot Encoding puede ser suficiente.
- Si hay categorias raras con muy pocos casos, podriamos agruparlas mas adelante.
- Debemos revisar si alguna categoria esta muy asociada con cancelacion.

## 11. Relacion inicial con el target

Comparamos algunas variables contra `booking_status`. Este paso nos ayuda a detectar senales predictivas iniciales.

In [ ]:
for column in categorical_columns + binary_columns:
    table = pd.crosstab(df[column], df[TARGET], normalize="index").round(3)
    print(f"\nProporcion por categoria: {column}")
    display(table)

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x=TARGET, y="lead_time")
plt.title("Lead time segun estado de reserva")
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x=TARGET, y="avg_price_per_room")
plt.title("Precio medio por habitacion segun estado de reserva")
plt.show()

Interpretacion a completar tras ejecutar:

- Si `lead_time` es mayor en reservas canceladas, podria ser una feature importante.
- Si algunas categorias tienen mayor tasa de cancelacion, pueden aportar senal al modelo.
- Estas observaciones son hipotesis: el modelo y las metricas las validaran despues.

## 12. Correlacion numerica

La correlacion solo mide relaciones lineales entre variables numericas. No captura todas las relaciones posibles, pero ayuda a detectar redundancias iniciales.

In [ ]:
plt.figure(figsize=(12, 8))
corr = df[numeric_columns].corr()
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Matriz de correlacion de variables numericas")
plt.show()

Interpretacion a completar tras ejecutar:

- Correlaciones altas entre features pueden indicar redundancia.
- Una correlacion baja no significa que una variable sea inutil.
- En clasificacion, tambien revisaremos relaciones con el target mediante metricas y modelos.

## 13. Conclusiones del EDA exploratorio

Completar despues de ejecutar el notebook:

- Calidad general de datos: TODO.
- Variables con outliers relevantes: TODO.
- Variables categoricas importantes: TODO.
- Variables candidatas a transformar: TODO.
- Variables a excluir adicionalmente: TODO.
- Riesgos detectados antes del preprocessing: TODO.

Proximo paso despues de cerrar este EDA: definir estrategia de preprocessing para baseline.